# Repaso: el flujo completo de un modelo de clasificación

Mapa de 10 pasos, de los datos crudos a la evaluación final. Cada paso lleva:
**qué hace** y **por qué va en ese lugar**.

El ejemplo usa **datos sintéticos** (no el dataset del banco), con un desbalance parecido (~88% "no" / ~12% "sí").
Así corre solo y puedes concentrarte en el orden de los pasos.

**Cómo usarlo:** léelo y corre las celdas en orden. Después explícame el flujo con tus palabras, aplicado al proyecto del banco.

**Tres reglas para quedarte:**
1. Lo que aprende de los datos va **después del split** y se ajusta **solo con train**.
2. Test se usa **una sola vez, al final**. No se usa para decidir.
3. Siempre compara contra el **modelo base**.

In [1]:
# Si te falta la librería, ejecútala UNA vez en la terminal (con el venv activo):
#   pip install imbalanced-learn

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

## Paso 1. Definir el problema y la métrica

**Qué hace:** fija qué predices y qué error cuesta más.
**Por qué va primero:** la métrica decide cómo evalúas todo lo demás.

En nuestro ejemplo: predecir si un cliente contrata (1) o no (0).
- **Falso negativo** (el cliente habría contratado y no lo llamas): pierdes una venta.
- **Falso positivo** (llamas a quien no contrata): cuesta una llamada.

Como perder ventas cuesta más y la clase "sí" es minoritaria, miramos **recall** y **AUC**, no la exactitud.

In [2]:
OBJETIVO = "objetivo"   # 1 = contrata, 0 = no contrata

## Paso 2. Explorar los datos (EDA)

**Qué hace:** forma, tipos, nulos y desbalance del objetivo.
**Por qué va antes de tocar nada:** necesitas entender qué tienes para decidir qué limpiar.

Los datos sintéticos incluyen tres "trampas" parecidas a las del banco:
- `dias_ult_contacto`: usa **-1** para "nunca contactado" (un código, no un número real).
- `duracion_llamada`: solo se conoce **después** de la llamada (fuga de información).
- `canal`: categórica de texto.

In [3]:
X_raw, y_raw = make_classification(
    n_samples=10_000, n_features=6, n_informative=4, n_redundant=0,
    weights=[0.88, 0.12], flip_y=0.02, random_state=42,
)
rng = np.random.default_rng(42)

df = pd.DataFrame(X_raw, columns=[f"var_{i}" for i in range(1, 7)])
df[OBJETIVO] = y_raw
df["canal"] = rng.choice(["movil", "fijo", "otro"], size=len(df), p=[0.6, 0.3, 0.1])

dias = rng.integers(10, 400, size=len(df)).astype(float)
dias[rng.random(len(df)) < 0.8] = -1          # 80% "nunca contactado"
df["dias_ult_contacto"] = dias

df["duracion_llamada"] = (df[OBJETIVO] * 200 + rng.normal(100, 50, len(df))).clip(lower=1)

print(df.shape)
print(df.dtypes)
print("\nNulos por columna:\n", df.isna().sum())
print("\nProporción del objetivo:\n", df[OBJETIVO].value_counts(normalize=True).round(3))

(10000, 10)
var_1                float64
var_2                float64
var_3                float64
var_4                float64
var_5                float64
var_6                float64
objetivo               int64
canal                    str
dias_ult_contacto    float64
duracion_llamada     float64
dtype: object

Nulos por columna:
 var_1                0
var_2                0
var_3                0
var_4                0
var_5                0
var_6                0
objetivo             0
canal                0
dias_ult_contacto    0
duracion_llamada     0
dtype: int64

Proporción del objetivo:
 objetivo
0    0.873
1    0.127
Name: proportion, dtype: float64


## Paso 3. Limpieza con reglas fijas

**Qué hace:** transformaciones que **no aprenden nada de los datos**. Cada fila se transforma con la misma regla, sin mirar a las demás.
**Por qué puede ir antes del split:** no hay nada que calcular con el dataset completo, así que no contamina la prueba.

Ejemplos (los mismos tipos que en tu script `preprocesamiento.py`):
- Crear el flag `contactado_antes` **antes** de tocar `dias_ult_contacto` (el flag necesita el `-1`).
- Pasar el `-1` a `NaN`.
- Quitar la fuga de información (`duracion_llamada`).

In [4]:
df_limpio = df.copy()

# 1) el flag primero: depende del -1
df_limpio["contactado_antes"] = (df_limpio["dias_ult_contacto"] != -1).astype(int)

# 2) luego el -1 pasa a NaN
df_limpio["dias_ult_contacto"] = df_limpio["dias_ult_contacto"].replace(-1, np.nan)

# 3) fuga de información: no estaría disponible antes de llamar
df_limpio = df_limpio.drop(columns=["duracion_llamada"])

print("NaN en dias_ult_contacto:", df_limpio["dias_ult_contacto"].isna().sum())
print(df_limpio["contactado_antes"].value_counts().to_dict())

NaN en dias_ult_contacto: 7957
{0: 7957, 1: 2043}


## Paso 4. Separar train y test

**Qué hace:** aparta una parte de los datos (test) que el modelo no ve durante el desarrollo.
**Por qué va antes de cualquier paso que aprenda de los datos:** si calculas medias, escalas o categorías con todo el dataset, el test ya "dejó huella" en el entrenamiento (*data leakage*).

- `stratify=y` conserva la proporción de "sí" en ambos lados.
- `random_state` hace que el resultado sea reproducible.

In [5]:
X = df_limpio.drop(columns=[OBJETIVO])
y = df_limpio[OBJETIVO]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Proporción de 'sí' → train:", y_train.mean().round(3), "| test:", y_test.mean().round(3))

Proporción de 'sí' → train: 0.127 | test: 0.127


## Paso 5. Preprocesamiento que aprende (solo con train)

**Qué hace:** imputar `NaN`, escalar, one-hot y SMOTE.
**Por qué después del split:** todos calculan algo a partir de los datos.

La regla de sklearn: **`fit` (calcula) solo con train; `transform` (aplica) con train y test.**

| Paso | Qué aprende en `fit` |
|---|---|
| Imputar con la mediana | la mediana de train |
| `StandardScaler` | media y desviación de train |
| `OneHotEncoder` | las categorías que ve en train |
| SMOTE | los vecinos de train (y **solo** se aplica a train) |

> Cómo tratar los `NaN` es una decisión tuya (imputar con un valor apoyándote en el flag `contactado_antes`, o usar un modelo que tolere `NaN`). Aquí se muestra **una** opción: la mediana.

In [6]:
num_cols = [c for c in X_train.columns if c.startswith("var_")]
cat_cols = ["canal"]

pre_dias = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

preprocesador = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("dias", pre_dias, ["dias_ult_contacto"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("bin", "passthrough", ["contactado_antes"]),
])

X_train_pre = preprocesador.fit_transform(X_train)   # fit: aprende SOLO de train
X_test_pre = preprocesador.transform(X_test)         # transform: aplica lo aprendido, no recalcula

print("train:", X_train_pre.shape, "| test:", X_test_pre.shape)

train: (8000, 11) | test: (2000, 11)


In [7]:
# SMOTE: crea ejemplos sintéticos de la clase minoritaria. SOLO en train.
X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(X_train_pre, y_train)

print("train antes :", y_train.value_counts().to_dict())
print("train SMOTE :", pd.Series(y_train_sm).value_counts().to_dict())
print("test (intacto):", y_test.value_counts().to_dict())

train antes : {0: 6985, 1: 1015}
train SMOTE : {0: 6985, 1: 6985}
test (intacto): {0: 1746, 1: 254}


## Paso 6. Modelo base

**Qué hace:** un modelo simple (regresión logística) que sirve de vara de comparación.
**Por qué va antes de los modelos complejos:** cualquier modelo más sofisticado tiene que superarlo, o no vale la pena su complejidad.

Para no repetir el preprocesamiento a mano, lo metemos en un `Pipeline`: así el `fit` del preprocesador se rehace **dentro de cada partición** de la validación cruzada, sin fugas.

In [8]:
pipe_base = Pipeline([
    ("pre", preprocesador),
    ("modelo", LogisticRegression(max_iter=1000)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Paso 7. Evaluar

**Qué hace:** mide cuánto acierta y **en qué se equivoca**.
**Por qué no basta la exactitud:** con ~88% de "no", un modelo que siempre dice "no" acierta ~88% y no sirve de nada.

Aquí evaluamos con **validación cruzada sobre train**. Test se queda guardado para el final.

- **Recall:** de los que sí contrataban, ¿a cuántos detectó?
- **Precisión:** de los que marcó como "sí", ¿cuántos eran "sí"?
- **AUC:** qué tan bien separa a los "sí" de los "no" en general (1 = perfecto, 0,5 = azar).

In [9]:
print("Exactitud de 'siempre digo no':", round(1 - y_train.mean(), 3))

proba_cv = cross_val_predict(pipe_base, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
pred_cv = (proba_cv >= 0.5).astype(int)

print("\nMatriz de confusión (filas = real, columnas = predicho):")
print(confusion_matrix(y_train, pred_cv))
print(classification_report(y_train, pred_cv, digits=3))
print("AUC:", round(roc_auc_score(y_train, proba_cv), 3))

Exactitud de 'siempre digo no': 0.873



Matriz de confusión (filas = real, columnas = predicho):
[[6967   18]
 [ 596  419]]
              precision    recall  f1-score   support

           0      0.921     0.997     0.958      6985
           1      0.959     0.413     0.577      1015

    accuracy                          0.923      8000
   macro avg      0.940     0.705     0.767      8000
weighted avg      0.926     0.923     0.909      8000

AUC: 0.78


## Paso 8. Mejorar

**Qué hace:** prueba alternativas y las compara **con la misma validación cruzada** y el modelo base como referencia.
**Por qué la validación cruzada es sobre train:** si decidieras mirando test, dejaría de ser una prueba independiente.

Alternativas del ejemplo: dar más peso a la clase minoritaria (`class_weight`), SMOTE y Random Forest.

> SMOTE va **dentro** del pipeline de `imblearn`: así se aplica solo a las particiones de entrenamiento de cada fold, y la partición de validación queda intacta.

In [10]:
candidatos = {
    "Logística (base)": pipe_base,
    "Logística + class_weight": Pipeline([
        ("pre", preprocesador),
        ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]),
    "Logística + SMOTE": ImbPipeline([
        ("pre", preprocesador),
        ("smote", SMOTE(random_state=42)),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]),
    "Random Forest": Pipeline([
        ("pre", preprocesador),
        ("modelo", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ]),
}

filas = []
for nombre, pipe in candidatos.items():
    auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    rec = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall")
    filas.append({"modelo": nombre, "AUC": auc.mean().round(3), "recall": rec.mean().round(3)})

tabla = pd.DataFrame(filas)
tabla

,modelo,AUC,recall
0,Logística (base),0.780,0.413
1,Logística + class_weight,0.811,0.698
2,Logística + SMOTE,0.811,0.697
3,Random Forest,0.936,0.633


## Paso 9. Elegir, ajustar el umbral y evaluar en test (una sola vez)

**Qué hace:** eliges el modelo según la tabla del paso 8, ajustas el **umbral de decisión** y recién ahí abres test.
**Por qué test al final y una sola vez:** si lo usas para decidir, deja de medir cómo se comportará el modelo con datos nuevos.

- El **umbral** (por defecto 0,5) es el corte de probabilidad para decir "sí". Bajarlo detecta más "sí" (sube el recall) a costa de más falsos positivos (baja la precisión).
- El umbral se elige con las **probabilidades de validación cruzada** (train). En test solo se **confirma**.
- Después vendrían la importancia de variables y la traducción a lenguaje de negocio.

In [11]:
mejor = candidatos["Random Forest"]   # ejemplo: el de mayor AUC en la tabla; en tu proyecto la elección sale de TU tabla

# 1) elegir el umbral con validación cruzada sobre train
proba_cv_mejor = cross_val_predict(mejor, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
for umbral in [0.5, 0.4, 0.3, 0.2]:
    p = (proba_cv_mejor >= umbral).astype(int)
    print(f"umbral {umbral}: recall={recall_score(y_train, p):.3f}  precisión={precision_score(y_train, p):.3f}")

umbral 0.5: recall=0.633  precisión=0.951
umbral 0.4: recall=0.693  precisión=0.902
umbral 0.3: recall=0.762  precisión=0.812
umbral 0.2: recall=0.829  precisión=0.676


In [12]:
UMBRAL = 0.3   # ejemplo: más recall sin que la precisión se desplome; en tu proyecto lo justificas tú

# 2) entrenar con todo train y abrir test UNA vez
mejor.fit(X_train, y_train)
proba_test = mejor.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= UMBRAL).astype(int)

print("Matriz de confusión en test:")
print(confusion_matrix(y_test, pred_test))
print(classification_report(y_test, pred_test, digits=3))
print("AUC en test:", round(roc_auc_score(y_test, proba_test), 3))

Matriz de confusión en test:
[[1704   42]
 [  61  193]]
              precision    recall  f1-score   support

           0      0.965     0.976     0.971      1746
           1      0.821     0.760     0.789       254

    accuracy                          0.949      2000
   macro avg      0.893     0.868     0.880      2000
weighted avg      0.947     0.949     0.948      2000

AUC en test: 0.928


## Paso 10. Empaquetar y desplegar (lo verás en diciembre y enero)

Solo el mapa, sin código por ahora:

- **MLflow:** registra parámetros, métricas y modelos de cada experimento, para comparar corridas y no perder lo que probaste.
- **FastAPI:** convierte el modelo en un servicio (`/predict`) que otros sistemas pueden consultar.
- **Docker:** empaqueta el servicio con todas sus dependencias para que corra igual en cualquier máquina.
- **AWS y monitoreo:** lo despliegas y revisas logs y predicciones para detectar problemas.

## Resumen: dónde estás en tu proyecto del banco

| Paso | Estado |
|---|---|
| 1. Problema y métrica | Hecho |
| 2. EDA | Hecho |
| 3. Limpieza con reglas fijas | Hecho (`src/preprocesamiento.py`) |
| 4. Split train/test | Hoy |
| 5. Preprocesamiento que aprende | Hoy (incluye la decisión sobre los `NaN` de `pdays`) |
| 6. Modelo base | Hoy |
| 7. Evaluar | Hoy y mañana |
| 8. Mejorar | Semanas 4 a 6 |
| 9. Elegir y explicar | Semanas 6 y 7 |
| 10. Empaquetar y desplegar | Diciembre y enero |

**Las tres reglas:** lo que aprende va después del split y solo con train · test una sola vez al final · compara siempre contra el modelo base.